# Harmonic-Oscillator MCW Demonstration

This notebook compares a Monte Carlo phase-space simulation with the
split-step Fourier solution of the one-dimensional Schrödinger equation.

For a Gaussian state in a quadratic potential, Wigner evolution follows
classical Hamiltonian flow. This makes the harmonic oscillator a useful
consistency check for sampling, propagation, and density reconstruction.

## Before running

From the repository's top-level folder, install the package with:

```bash
python3 -m pip install -e .
```

Then restart the Jupyter kernel and run this notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mcw import (
    gaussian_wavepacket,
    split_operator_propagate,
    sample_wigner_gaussian,
    gaussian_kde_marginal,
)
from mcw.plots import compare_densities

from mcw import propagate_harmonic


In [ ]:
hbar = 1.0
mass = 1.0
omega = 0.3

dt = 0.02
total_time = 12.0
nsteps = round(total_time / dt)
steps = {0, round(4 / dt), round(8 / dt), nsteps}

x = np.linspace(-40.0, 40.0, 2**12, endpoint=False)
potential = lambda grid: 0.5 * mass * omega**2 * grid**2

psi0 = gaussian_wavepacket(
    x,
    x0=-10.0,
    p0=3.0,
    sigma=1.5,
    hbar=hbar,
)


In [ ]:
psi_snapshots = split_operator_propagate(
    psi0,
    x,
    dt,
    nsteps,
    potential,
    mass=mass,
    hbar=hbar,
    snapshot_steps=steps,
)

xs0, ps0, signs = sample_wigner_gaussian(
    40_000,
    x0=-10.0,
    p0=3.0,
    sigma=1.5,
    hbar=hbar,
    seed=123,
)

particle_snapshots = propagate_harmonic(
    xs0,
    ps0,
    dt,
    nsteps,
    mass=mass,
    omega=omega,
    snapshot_steps=steps,
)


In [ ]:
for step in sorted(steps):
    reference = np.abs(psi_snapshots[step]) ** 2
    reference /= np.trapz(reference, x)

    xs, _ = particle_snapshots[step]
    estimate = gaussian_kde_marginal(
        xs,
        signs,
        x,
        bandwidth=0.15,
    )

    compare_densities(
        x,
        reference,
        estimate,
        title=f"Harmonic oscillator, t={step * dt:.2f}",
    )
    plt.show()
